# 03 · Train all models on Google Colab: Solar-Energy

Dinesh's notebook for **Solar** (Chandana's Electricity version: `03_train_on_colab.ipynb`).

Open in Colab:
`https://colab.research.google.com/github/Chandana18G/lstnet-reproduction/blob/main/notebooks/03_train_on_colab_solar.ipynb`

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells from top to bottom. Nothing to change: the dataset is fixed to Solar.

Everything is saved to **Google Drive** (`MyDrive/lstnet-checkpoints`) after every epoch.
If Colab stops, run all cells again: finished runs are skipped and an unfinished run continues
where it stopped. You can also split the work over several days.

## 1 · GPU check

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

## 2 · Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
CKPT = "/content/drive/MyDrive/lstnet-checkpoints"
import os
os.makedirs(CKPT, exist_ok=True)

## 3 · Code and data

Solar settings as in the authors' `solar.sh`: `--hid-skip 10`; horizon 24 as in the paper's
table (the script itself runs horizon 12, see `CLAUDE.md`).

In [ ]:
DATA = "solar"
EXTRA = "--hid-skip 10"         # authors' solar.sh; Electricity uses the default 5
%cd /content
!git clone -q https://github.com/Chandana18G/lstnet-reproduction.git || (cd lstnet-reproduction && git pull -q)
%cd /content/lstnet-reproduction
!bash scripts/download_data.sh {DATA}

## 4 · First: one LSTNet run (seed 0)

Run this first to see how long one run takes and whether we get close to the paper
(Solar: test RSE 0.4643, CORR 0.8870; our AR baseline: 0.5247). The first printed line must be
`run solar_h24_lstnet_skip24_seed0 on cuda`. Look at the time per epoch: 100 epochs × that time
= one run.

In [ ]:
import shutil, os
def run(model, seed):
    name = f"{DATA}_h24_{model}_skip24_seed{seed}"
    if os.path.exists(f"{CKPT}/{name}.json"):
        print("already done:", name)
        return
    get_ipython().system(f"python src/train.py --data {DATA} --horizon 24 --model {model} --seed {seed} {EXTRA} --checkpoint-dir {CKPT}")
    shutil.copy(f"results/runs/{name}.json", CKPT)      # keep the result on Drive

run("lstnet", 0)

## 5 · All models, seeds 0 to 9

Order: seed by seed, so that after any stop you have complete pairs (same seed for every
model). `lstnet` and `lstnet_revin` are the main comparison; `dlinear` is the simple linear
model; `lstnet_noar` is the ablation without AR.

In [ ]:
for seed in range(10):
    for model in ["lstnet", "lstnet_revin", "dlinear", "lstnet_noar"]:
        run(model, seed)

## 6 · AR baseline and statistics

The AR baseline (needed for one comparison) runs on the CPU and takes a while for Solar
(L up to 512). Then the statistics: mean ± std over seeds, paired tests over seeds, Diebold-Mariano.

In [ ]:
!mkdir -p results/runs && cp {CKPT}/{DATA}_*.json results/runs/
!python src/baselines.py --data {DATA} --horizon 24
!python src/stats.py --data {DATA} --horizon 24 --pred-dir {CKPT}
shutil.copy(f"results/stats_{DATA}_h24.md", CKPT)

## 7 · Bring the results home

From `MyDrive/lstnet-checkpoints` download all `solar_*.json` files into `results/runs/` and
`stats_solar_h24.md` into `results/` of your local repo, then commit. The
`_test_predictions.npy` and `.pt` files stay on Drive (too big for git).